# Lecture 3 — Exercise: Scatter & Bubble Charts
Dataset: Gapminder (built into plotly).

In [4]:
%pip install plotly


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 1.2 MB/s  0:00:08 eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = px.data.gapminder()
print(df.shape)
print(df.columns.tolist())
df.head()

(1704, 8)
['country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap', 'iso_alpha', 'iso_num']


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


## Task 1 — Scatter: how Asia moved between 2002 and 2007
I chose **Asia**. Colour = year (only 2 colours), and a thin arrow joins each country's 2002 point to its 2007 point so the direction of change is visible.

In [8]:
asia = df.loc[(df['continent'] == 'Asia') & (df['year'].isin([2002, 2007]))]

# put 2002 and 2007 side by side so I can draw one arrow per country
wide = asia.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])

C2002, C2007 = '#9AA5B1', '#C0392B'   # grey-blue for the start, red for the end

fig = go.Figure()

# arrows from 2002 -> 2007
for country in wide.index:
    fig.add_annotation(
        x=wide.loc[country, ('gdpPercap', 2007)], y=wide.loc[country, ('lifeExp', 2007)],
        ax=wide.loc[country, ('gdpPercap', 2002)], ay=wide.loc[country, ('lifeExp', 2002)],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1, arrowcolor='#BBBBBB'
    )

# the two years as points
for yr, col in [(2002, C2002), (2007, C2007)]:
    d = asia[asia['year'] == yr]
    fig.add_trace(go.Scatter(
        x=d['gdpPercap'], y=d['lifeExp'], mode='markers', name=str(yr),
        marker=dict(color=col, size=10),
        text=d['country'],
        hovertemplate='%{text}<br>GDP/cap: %{x:,.0f}<br>Life exp: %{y:.1f}<extra></extra>'
    ))

# label a few interesting countries instead of all of them
for c in ['Iraq', 'Kuwait', 'Japan', 'China', 'Afghanistan']:
    fig.add_annotation(x=wide.loc[c, ('gdpPercap', 2007)], y=wide.loc[c, ('lifeExp', 2007)],
                       text=c, showarrow=False, xshift=6, yshift=10,
                       xanchor='left', font=dict(size=11, color='#444'))

fig.update_layout(
    title=dict(text="<b>Between 2002 and 2007 almost every Asian country moved up and to the right</b>"
                    "<br><sup>Richer and longer-living; arrows show the change per country</sup>",
               x=0, xanchor='left'),
    template='simple_white',
    xaxis=dict(title='GDP per capita (US$)', tickformat=',', showgrid=False),
    yaxis=dict(title='Life expectancy (years)', showgrid=False),
    legend=dict(title='Year', orientation='h', y=1.02, x=1, xanchor='right', yanchor='bottom'),
    height=520, margin=dict(t=100)
)
fig.show(renderer="browser")

## Task 2 — Bubble chart, 2007: Asia highlighted
x = GDP per capita (log), y = life expectancy, size = population. Asia in colour, all other continents in grey.

In [10]:
%pip install --upgrade nbformat ipython

  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [nbformat]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import nbformat
print(nbformat.__version__)

5.11.1


In [13]:
d07 = df[df['year'] == 2007].copy()
HL = 'Asia'
d07['group'] = d07['continent'].where(d07['continent'] == HL, 'Other continents')

fig = px.scatter(
    d07, x='gdpPercap', y='lifeExp', size='pop', color='group',
    color_discrete_map={HL: '#C0392B', 'Other continents': '#DDDDDD'},
    hover_name='country', size_max=60, log_x=True,
    category_orders={'group': ['Other continents', HL]},   # grey drawn first, Asia on top
    labels=dict(gdpPercap='GDP per capita (US$, log scale)', lifeExp='Life expectancy (years)')
)
fig.update_traces(marker=dict(line=dict(width=0.5, color='white')))

# annotation: the story
fig.add_annotation(
    x=d07.loc[d07['country'] == 'China', 'gdpPercap'].iloc[0],
    y=d07.loc[d07['country'] == 'China', 'lifeExp'].iloc[0],
    text='<b>China & India</b><br>over 2 billion people together,<br>mid income, life exp. ~65-73',
    ax=-110, ay=-90, showarrow=True, arrowhead=2, arrowcolor='#C0392B',
    font=dict(color='#C0392B', size=12), align='left'
)
fig.add_annotation(
    x=d07.loc[d07['country'] == 'Japan', 'gdpPercap'].iloc[0],
    y=d07.loc[d07['country'] == 'Japan', 'lifeExp'].iloc[0],
    text='Japan: highest life expectancy', ax=-30, ay=-45, showarrow=True,
    arrowhead=2, arrowcolor='#C0392B', font=dict(color='#C0392B', size=11)
)

fig.update_layout(
    title=dict(text="<b>Asia is the most spread-out continent in 2007</b>"
                    "<br><sup>From Afghanistan (43 yrs) to Japan (83 yrs), and it holds the world's biggest populations</sup>",
               x=0, xanchor='left'),
    template='simple_white', showlegend=False, height=580, margin=dict(t=100),
    xaxis=dict(showgrid=False), yaxis=dict(showgrid=False)
)
fig.show(renderer="browser")